# A03 — DBR mirror

## Mathematical and physical model

Use time dependence $e^{-i\omega t}$, vacuum wavenumber $k_0=2\pi/\lambda_0$ and conserved transverse wavevector $k_\parallel$. For each isotropic, nonmagnetic layer set
$$
u_j=k_{z,j}/k_0=\sqrt{\epsilon_j-(k_\parallel/k_0)^2},\qquad
Y_{s,j}=u_j,\qquad Y_{p,j}=\epsilon_j/u_j .
$$
Choose the forward branch with $\operatorname{Im}u_j\ge 0$ for passive layers. The admittances apply to **tangential** electric-field coefficients. The notebook compares the solver outputs only after converting them to the common ray-based polarization and power conventions.

For $\delta_j=k_0u_jd_j$, the film matrix mapping the tangential state from the exit plane to the entrance plane is
$$
M_j=\begin{pmatrix}\cos\delta_j & -i\sin\delta_j/Y_j\\-iY_j\sin\delta_j & \cos\delta_j\end{pmatrix}.
$$
The ordered stack is $(M_H M_L)^8$, with $\delta_H=\delta_L=\pi/2$ at the 1550 nm design wavelength at normal incidence. The external admittances set the radiation boundary conditions.

## Physical scenario

An eight-pair stack alternates high- and low-index dielectric films, each with one-quarter-wave optical thickness at 1550 nm. Reflections from successive interfaces add coherently near the design wavelength and produce a stopband. Sweeping through the stopband and its edges probes layer order and reflection phase.

## Possible applications

Represents a distributed Bragg reflector used as a laser cavity mirror, an optical filter element, or a high-reflectance coating for near-infrared photonics.

## Benchmark objective

Check layer order, stopband and phase for eight quarter-wave dielectric pairs at a design wavelength of 1550 nm.

## Reproduce

Use the project environment and select **Run All**. The notebook and CLI use [case.yaml](case.yaml). The native [Lumerical model](model.fsp) and [reconstruction script](model.lsf) correspond to quick-sweep point 0.

[Published comparisons and figures](../../../BENCHMARK_REPORT.md#a03)

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

from rcwa_cv.case_loader import catalog
from rcwa_cv.notebook import case_setup, show_results, show_versions, solve_sweep
from rcwa_cv.reporting.plots import geometry_figure

case_path = catalog("A03")[0]
samples, settings, metadata = case_setup(case_path)
results = {}

print(f"{len(samples)} points, Fourier truncation index {settings.order}")
figure = geometry_figure(case_path)
display(figure)
plt.close(figure)

## Analytic multilayer reference

In [ ]:
from rcwa_cv.references.multilayer import solve as solve_reference

results["reference"] = solve_sweep(solve_reference, samples, settings)

## meent · NumPy CPU

In [ ]:
from rcwa_cv.solvers.meent_cpu import probe, solve

print(probe())
results["meent"] = solve_sweep(solve, samples, settings)

## torcwa · CUDA GPU, complex128

In [ ]:
from rcwa_cv.solvers.torcwa_cuda import probe, solve

print(probe())
results["torcwa"] = solve_sweep(solve, samples, settings)

## grcwa · NumPy CPU

In [ ]:
from rcwa_cv.solvers.grcwa_cpu import probe, solve

print(probe())
results["grcwa"] = solve_sweep(solve, samples, settings)

## Lumerical RCWA

This cell uses the installed, licensed FDTD application through its Python API. It reuses one session
across the sweep and evaluates the same model generator used by the reconstruction script.

In [ ]:
from rcwa_cv.solvers.lumerical import session, solve

with session() as fdtd:
    print(f"Lumerical build {fdtd.version()}")
    results["lumerical"] = solve_sweep(solve, samples, settings)

## Compare the solvers

The spectra show total reflection, transmission and the co-polarized transmission phase.
The order plot uses a point shared by the solvers. The heatmaps show maximum discrepancies
over the tested points. Periodic quick sweeps provide preliminary comparisons.

In [ ]:
show_versions(results)
show_results(metadata, results)